# Automatic News Headline Sentiment Categorizer

**Topics:** Natural Language Understanding (NLU), Sentiment Analysis, Sequence Labeling, Parsing, Ambiguity Analysis  
**Goal:** Build a multi-class classifier that automatically categorizes news headlines by **sentiment** (Positive / Negative / Neutral) using linguistic features and logical reasoning. Includes detailed analysis of sentiment ambiguity through misclassified examples.

**Dataset:** ABC News Headlines (~1.2M headlines, 2003–2021)  
**Format:** `publish_date` (yyyymmdd) | `headline_text`  

**Approach:**
1. Label headlines using VADER sentiment analysis (ground-truth generation).
2. Analyze linguistic patterns (POS tags, named entities) across sentiment classes.
3. Train ML classifiers (TF-IDF + SVM/LR/NB/RF) to learn sentiment patterns.
4. Analyze ambiguous/misclassified cases through logical reasoning.

---

## Setup & Imports

In [ ]:
import sys
import os
import warnings
warnings.filterwarnings('ignore')

# Add project root to path so we can import our src modules
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import spacy
from collections import Counter

from sklearn.model_selection import train_test_split

# Import our custom modules
from src.preprocess import (
    load_headline_dataset, label_sentiment_vader,
    load_spacy_model, preprocess_batch, extract_entity_features,
)
from src.features import (
    get_pos_distribution, get_top_words_per_sentiment, build_tfidf_vectorizer,
)
from src.model import (
    build_pipeline, evaluate_model, get_misclassified, get_top_features_per_class,
)

# Plot styling
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print(f"Project root: {PROJECT_ROOT}")
print("Setup complete.")

---
## Step 1: Load & Explore the Dataset

The ABC News Headlines dataset contains ~1.2 million headlines with publish dates. We sample a manageable subset for analysis while preserving temporal distribution.

In [ ]:
# Load the ABC News headlines dataset
# We sample 100,000 headlines for tractable processing; increase for more coverage
DATA_PATH = os.path.join(PROJECT_ROOT, 'data', 'abcnews-date-text.csv')
SAMPLE_SIZE = 100_000

df = load_headline_dataset(DATA_PATH, sample_size=SAMPLE_SIZE, random_state=RANDOM_STATE)

print(f"\nDataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"Date range: {df['publish_date'].min().date()} to {df['publish_date'].max().date()}")
print(f"\nMissing values:\n{df.isnull().sum()}")
df.head(10)

In [ ]:
# Add temporal features for analysis
df['year'] = df['publish_date'].dt.year
df['month'] = df['publish_date'].dt.month
df['word_count'] = df['headline_text'].apply(lambda x: len(x.split()))

# Headline length distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['word_count'], bins=30, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_title('Headline Word Count Distribution', fontweight='bold', fontsize=14)
axes[0].set_xlabel('Word Count')
axes[0].set_ylabel('Frequency')
axes[0].axvline(df['word_count'].median(), color='red', linestyle='--', label=f"Median: {df['word_count'].median():.0f}")
axes[0].legend()

# Headlines per year
year_counts = df['year'].value_counts().sort_index()
axes[1].bar(year_counts.index, year_counts.values, color='coral', edgecolor='white')
axes[1].set_title('Headlines per Year (sampled)', fontweight='bold', fontsize=14)
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

print(f"\nWord count stats: mean={df['word_count'].mean():.1f}, "
      f"median={df['word_count'].median():.0f}, max={df['word_count'].max()}")

In [ ]:
# Show random sample headlines
print("Random sample of 15 headlines:")
print("=" * 70)
for _, row in df.sample(15, random_state=RANDOM_STATE).iterrows():
    print(f"  [{row['publish_date'].strftime('%Y-%m-%d')}] {row['headline_text']}")

### Top 20 Most Common Words in Headlines

Before any sentiment labeling, we extract the **most frequent meaningful words** across all sampled headlines.  
This reveals the dominant topics and vocabulary that the dataset revolves around — and gives insight into what the classifier will eventually work with.

In [ ]:
# Top 20 most common words across all headlines (stopword-filtered)
from collections import Counter
import re

# Simple but effective: tokenize, lowercase, remove stopwords and short tokens
STOP_WORDS = set(nlp.Defaults.stop_words) if 'nlp' in dir() else set()
if not STOP_WORDS:
    _nlp_temp = spacy.blank('en')
    STOP_WORDS = _nlp_temp.Defaults.stop_words

def get_top_words(texts, stop_words, top_n=20):
    """Extract top N most frequent meaningful words from texts."""
    word_counter = Counter()
    for text in texts:
        tokens = re.findall(r'\b[a-z]{3,}\b', text.lower())
        tokens = [t for t in tokens if t not in stop_words]
        word_counter.update(tokens)
    return word_counter.most_common(top_n)

top20 = get_top_words(df['headline_text'].tolist(), STOP_WORDS, top_n=20)

# Display as table
print("TOP 20 MOST COMMON WORDS IN HEADLINES")
print("=" * 50)
for rank, (word, count) in enumerate(top20, 1):
    bar = '█' * (count // max(1, top20[-1][1] // 20))
    print(f"  {rank:2d}. {word:<18s} {count:>6,d}  {bar}")

# Bar chart
fig, ax = plt.subplots(figsize=(12, 7))
words = [w for w, _ in top20]
counts = [c for _, c in top20]
bars = ax.barh(words[::-1], counts[::-1], color=plt.cm.viridis(np.linspace(0.3, 0.9, 20)),
               edgecolor='white', linewidth=0.5)
ax.set_xlabel('Frequency', fontsize=12)
ax.set_title('Top 20 Most Common Words in Headlines', fontweight='bold', fontsize=14)
for bar, count in zip(bars, counts[::-1]):
    ax.text(bar.get_width() + max(counts) * 0.01, bar.get_y() + bar.get_height() / 2,
            f'{count:,}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'results', 'top20_common_words.png'),
            dpi=150, bbox_inches='tight')
plt.show()

all_words = Counter(t for text in df['headline_text'] for t in re.findall(r'\b[a-z]{3,}\b', text.lower()) if t not in STOP_WORDS)
print(f"\nTotal unique words (after filtering): {len(all_words):,}")
print(f"Total word occurrences: {sum(all_words.values()):,}")

In [ ]:
# Detailed headline statistics
df['char_count'] = df['headline_text'].apply(len)

print("DETAILED HEADLINE STATISTICS")
print("=" * 60)
print(f"  Total headlines sampled:    {len(df):,}")
print(f"  Unique headlines:           {df['headline_text'].nunique():,}")
print(f"  Duplicate headlines:        {len(df) - df['headline_text'].nunique():,}")
print(f"  Date range:                 {df['publish_date'].min().date()} → {df['publish_date'].max().date()}")
print(f"  Number of unique years:     {df['year'].nunique()}")
print(f"\n  --- Word Count ---")
print(f"  Mean words per headline:    {df['word_count'].mean():.1f}")
print(f"  Median words per headline:  {df['word_count'].median():.0f}")
print(f"  Min words:                  {df['word_count'].min()}")
print(f"  Max words:                  {df['word_count'].max()}")
print(f"  Std deviation:              {df['word_count'].std():.2f}")
print(f"\n  --- Character Count ---")
print(f"  Mean chars per headline:    {df['char_count'].mean():.1f}")
print(f"  Median chars per headline:  {df['char_count'].median():.0f}")
print(f"  Min chars:                  {df['char_count'].min()}")
print(f"  Max chars:                  {df['char_count'].max()}")

# Shortest and longest headlines
print(f"\n  --- Shortest Headlines (by word count) ---")
for _, row in df.nsmallest(5, 'word_count').iterrows():
    print(f"    [{row['word_count']} words] {row['headline_text']}")
print(f"\n  --- Longest Headlines (by word count) ---")
for _, row in df.nlargest(5, 'word_count').iterrows():
    print(f"    [{row['word_count']} words] {row['headline_text']}")

# Character length distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['char_count'], bins=40, color='#8e44ad', edgecolor='white', alpha=0.8)
axes[0].set_title('Character Count Distribution', fontweight='bold', fontsize=14)
axes[0].set_xlabel('Character Count')
axes[0].set_ylabel('Frequency')
axes[0].axvline(df['char_count'].median(), color='red', linestyle='--',
                label=f"Median: {df['char_count'].median():.0f}")
axes[0].legend()

# Word count vs character count scatter (sampled)
scatter_sample = df.sample(min(5000, len(df)), random_state=RANDOM_STATE)
axes[1].scatter(scatter_sample['word_count'], scatter_sample['char_count'],
                alpha=0.15, s=8, color='#2980b9')
axes[1].set_title('Word Count vs Character Count', fontweight='bold', fontsize=14)
axes[1].set_xlabel('Word Count')
axes[1].set_ylabel('Character Count')

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'results', 'headline_stats.png'),
            dpi=150, bbox_inches='tight')
plt.show()

---
## Step 2: Sentiment Labeling with VADER

Since the dataset has no pre-existing sentiment labels, we use **NLTK's VADER** (Valence Aware Dictionary and sEntiment Reasoner) to generate ground-truth sentiment labels. VADER is specifically tuned for social media and news text, making it ideal for headline analysis.

**Labeling logic:**
- Compound score > 0.05 → **Positive**
- Compound score < −0.05 → **Negative**
- Otherwise → **Neutral**

In [ ]:
# Apply VADER sentiment labeling
print("Labeling sentiment with VADER...")
labels, scores = label_sentiment_vader(df['headline_text'].tolist())
df['sentiment'] = labels
df['vader_score'] = scores
print("Done.")

print(f"\nSentiment distribution:")
print(df['sentiment'].value_counts())
print(f"\nVADER compound score stats:")
print(df['vader_score'].describe().round(4))

In [ ]:
# Sentiment distribution visualizations
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Sentiment count bar plot
sent_counts = df['sentiment'].value_counts().reindex(['negative', 'neutral', 'positive'])
colors_sent = {'negative': '#e74c3c', 'neutral': '#95a5a6', 'positive': '#2ecc71'}
bars = axes[0].bar(sent_counts.index, sent_counts.values,
                   color=[colors_sent[s] for s in sent_counts.index],
                   edgecolor='gray', linewidth=0.5)
axes[0].set_title('Sentiment Distribution', fontweight='bold', fontsize=14)
axes[0].set_ylabel('Number of Headlines')
for bar, val in zip(bars, sent_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
                 f"{val:,}", ha='center', fontweight='bold')

# 2. VADER score distribution histogram
for sent, color in colors_sent.items():
    subset = df[df['sentiment'] == sent]['vader_score']
    axes[1].hist(subset, bins=50, alpha=0.5, color=color, label=sent, density=True)
axes[1].set_title('VADER Compound Score Distribution', fontweight='bold', fontsize=14)
axes[1].set_xlabel('Compound Score')
axes[1].set_ylabel('Density')
axes[1].axvline(0.05, color='green', linestyle='--', alpha=0.5)
axes[1].axvline(-0.05, color='red', linestyle='--', alpha=0.5)
axes[1].legend()

# 3. Proportion pie chart
axes[2].pie(sent_counts.values, labels=sent_counts.index, autopct='%1.1f%%',
            colors=[colors_sent[s] for s in sent_counts.index],
            startangle=140, textprops={'fontsize': 12})
axes[2].set_title('Sentiment Proportion', fontweight='bold', fontsize=14)

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'results', 'sentiment_distribution.png'),
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Show examples per sentiment class
for sent in ['positive', 'neutral', 'negative']:
    print(f"\n{'='*70}")
    print(f"  {sent.upper()} headlines (5 examples)")
    print(f"{'='*70}")
    subset = df[df['sentiment'] == sent].sample(5, random_state=RANDOM_STATE)
    for _, row in subset.iterrows():
        print(f"  [{row['vader_score']:+.4f}] {row['headline_text']}")

In [ ]:
# Sentiment trends over time
sent_by_year = df.groupby(['year', 'sentiment']).size().unstack(fill_value=0)
sent_by_year_pct = sent_by_year.div(sent_by_year.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(14, 6))
sent_by_year_pct[['negative', 'neutral', 'positive']].plot(
    kind='area', stacked=True, ax=ax,
    color=['#e74c3c', '#95a5a6', '#2ecc71'], alpha=0.7,
)
ax.set_title('Sentiment Proportion Over Time', fontweight='bold', fontsize=14)
ax.set_xlabel('Year')
ax.set_ylabel('Percentage (%)')
ax.legend(title='Sentiment', loc='upper right')
ax.set_ylim(0, 100)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'results', 'sentiment_over_time.png'),
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Train/test split with stratification on sentiment
X_train, X_test, y_train, y_test = train_test_split(
    df['headline_text'].tolist(),
    df['sentiment'].tolist(),
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=df['sentiment'],
)

print(f"Training set: {len(X_train):,} headlines")
print(f"Test set:     {len(X_test):,} headlines")
print(f"\nTrain sentiment distribution:")
print(pd.Series(y_train).value_counts().sort_index())
print(f"\nTest sentiment distribution:")
print(pd.Series(y_test).value_counts().sort_index())

---
## Step 3: NLP Pipeline with SpaCy

We use SpaCy for:
- Tokenization & lemmatization
- Stopword & punctuation removal
- Named Entity Recognition (NER) — to understand what types of entities appear in each sentiment class

In [ ]:
# Load SpaCy model
nlp = load_spacy_model('en_core_web_sm')
print(f"SpaCy model loaded: {nlp.meta['name']} (v{nlp.meta['version']})")
print(f"Pipeline components: {nlp.pipe_names}")
nlp.max_length = 2_000_000

In [ ]:
# Preprocess training and test headlines
print("Preprocessing training headlines...")
X_train_clean = preprocess_batch(X_train, nlp, batch_size=500)
print(f"  Done. Sample: '{X_train_clean[0]}'")

print("Preprocessing test headlines...")
X_test_clean = preprocess_batch(X_test, nlp, batch_size=500)
print(f"  Done. Sample: '{X_test_clean[0]}'")

# Original vs cleaned comparison
print("\n--- Comparison (first 5) ---")
for orig, clean in list(zip(X_train[:5], X_train_clean[:5])):
    print(f"  Original: {orig}")
    print(f"  Cleaned:  {clean}")
    print()

In [ ]:
# Named Entity Recognition (NER) analysis per sentiment
# Use a subset for NER (computationally intensive)
NER_SAMPLE = 10_000
ner_idx = np.random.choice(len(X_train), min(NER_SAMPLE, len(X_train)), replace=False)
ner_texts = [X_train[i] for i in ner_idx]
ner_labels = [y_train[i] for i in ner_idx]

print(f"Extracting named entities from {len(ner_texts):,} headlines...")
ent_df = extract_entity_features(ner_texts, nlp)
ent_df['sentiment'] = ner_labels

# Average entity counts per sentiment
ent_cols = [c for c in ent_df.columns if c.startswith('ent_')]
ent_means = ent_df.groupby('sentiment')[ent_cols].mean().round(3)
ent_means.columns = [c.replace('ent_', '') for c in ent_means.columns]

print("\nAverage Named Entity counts per sentiment class:")
print(ent_means.to_string())

# Heatmap
fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(ent_means.T, annot=True, fmt='.3f', cmap='YlOrRd', ax=ax,
            linewidths=0.5, cbar_kws={'label': 'Avg Count'})
ax.set_title('Average Named Entity Counts by Sentiment', fontweight='bold', fontsize=14)
ax.set_xlabel('Sentiment')
ax.set_ylabel('Entity Type')
plt.tight_layout()
plt.show()

print("\nKey observations:")
print("- Negative headlines tend to have more GPE entities (places involved in crises, conflicts).")
print("- Positive headlines often include PERSON entities (achievements, awards).")
print("- Neutral headlines are dominated by ORG and GPE (factual reporting).")

---
## Step 4: Sequence Labeling & POS Analysis

We analyze Part-of-Speech (POS) tag distributions across sentiment classes to uncover linguistic reasoning patterns:
- **Negative** headlines: more adjectives ("deadly", "worst"), verbs ("kills", "crashes")
- **Positive** headlines: positive adjectives ("great", "best"), achievement verbs ("wins", "celebrates")
- **Neutral** headlines: more nouns and proper nouns (factual, entity-heavy)

In [ ]:
# POS tag distribution analysis
POS_SAMPLE = 10_000
pos_idx = np.random.choice(len(X_train), min(POS_SAMPLE, len(X_train)), replace=False)
pos_texts = [X_train[i] for i in pos_idx]
pos_labels = [y_train[i] for i in pos_idx]

print(f"Analyzing POS tags for {len(pos_texts):,} headlines...")
pos_df = get_pos_distribution(pos_texts, nlp, pos_labels)
print("Done.")

# Focus on informative POS tags
key_pos = ['NOUN', 'PROPN', 'VERB', 'ADJ', 'ADV', 'NUM']
pos_agg = pos_df.groupby(['sentiment', 'pos_tag'])['count'].sum().reset_index()
pos_filtered = pos_agg[pos_agg['pos_tag'].isin(key_pos)].copy()

# Normalize to proportions per sentiment
sent_totals = pos_filtered.groupby('sentiment')['count'].sum()
pos_filtered['proportion'] = pos_filtered.apply(
    lambda row: row['count'] / sent_totals[row['sentiment']], axis=1
)

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(data=pos_filtered, x='pos_tag', y='proportion', hue='sentiment',
            palette={'negative': '#e74c3c', 'neutral': '#95a5a6', 'positive': '#2ecc71'},
            ax=ax)
ax.set_title('POS Tag Proportions by Sentiment', fontweight='bold', fontsize=14)
ax.set_xlabel('POS Tag')
ax.set_ylabel('Proportion')
ax.legend(title='Sentiment', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'results', 'pos_distribution.png'),
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Top words per sentiment class
print("Extracting top words per sentiment...")
top_words = get_top_words_per_sentiment(pos_texts, pos_labels, nlp, top_n=15)

for sent, words in top_words.items():
    print(f"\n{'─'*55}")
    print(f"  {sent.upper()} — Top 15 Words (Nouns/Verbs/Adj/PropN)")
    print(f"{'─'*55}")
    for rank, (word, count) in enumerate(words, 1):
        bar = '█' * (count // 5)
        print(f"  {rank:2d}. {word:<20s} {count:>5d}  {bar}")

# Visualize top 8 words per sentiment
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
color_map = {'negative': '#e74c3c', 'neutral': '#95a5a6', 'positive': '#2ecc71'}
for ax, sent in zip(axes, ['negative', 'neutral', 'positive']):
    if sent in top_words:
        top8 = top_words[sent][:8]
        labels = [w for w, _ in top8]
        counts = [c for _, c in top8]
        ax.barh(labels[::-1], counts[::-1], color=color_map[sent], edgecolor='white')
        ax.set_title(f'{sent.upper()}', fontweight='bold', fontsize=13)
        ax.set_xlabel('Count')

plt.suptitle('Top 8 Words per Sentiment Class', fontweight='bold', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

### Linguistic Reasoning: POS & NER Insights

| Sentiment | Linguistic Pattern | Reasoning |
|---|---|---|
| **Negative** | More ADJ (deadly, worst, critical), action VERB (kill, crash, attack), GPE entities | Negative events are described with strong adjectives and action verbs; crises are geographically anchored |
| **Positive** | Achievement VERB (win, boost, celebrate), positive ADJ (new, great, top), PERSON entities | Positive headlines focus on people's achievements and use aspirational language |
| **Neutral** | Dominated by NOUN and PROPN, fewer ADJ, more ORG entities | Neutral headlines are factual, entity-dense, and descriptive without valence |

---
## Step 5: Text Vectorization with TF-IDF + N-grams

In [ ]:
# Build TF-IDF vectorizer on cleaned text
tfidf = build_tfidf_vectorizer(
    ngram_range=(1, 2),
    max_df=0.8,
    min_df=5,
    max_features=10000,
    use_stop_words=False,  # already cleaned
)

# Fit on training data
X_train_tfidf = tfidf.fit_transform(X_train_clean)
X_test_tfidf = tfidf.transform(X_test_clean)

print(f"TF-IDF matrix shape (train): {X_train_tfidf.shape}")
print(f"TF-IDF matrix shape (test):  {X_test_tfidf.shape}")
print(f"Number of features: {len(tfidf.get_feature_names_out())}")
print(f"\nSample unigrams: {list(tfidf.get_feature_names_out()[:15])}")
print(f"Sample bigrams:  {[f for f in tfidf.get_feature_names_out() if ' ' in f][:15]}")

---
## Step 6: Train and Evaluate Classifiers

We compare four classifiers to learn the mapping from TF-IDF features to sentiment:
1. **Linear SVM** (LinearSVC) — excellent for high-dimensional sparse text
2. **Logistic Regression** — strong baseline with probability output
3. **Multinomial Naive Bayes** — classic text classification baseline
4. **Random Forest** — ensemble comparison

In [ ]:
# Train and evaluate all four models
SENTIMENTS = sorted(df['sentiment'].unique())  # ['negative', 'neutral', 'positive']
model_names = ['linearsvc', 'logistic', 'naivebayes', 'randomforest']
results = {}

for name in model_names:
    print(f"\n{'='*60}")
    print(f"  Training: {name.upper()}")
    print(f"{'='*60}")
    
    vec = build_tfidf_vectorizer(
        ngram_range=(1, 2), max_df=0.8, min_df=5,
        max_features=10000, use_stop_words=False,
    )
    pipeline = build_pipeline(vec, classifier_name=name, random_state=RANDOM_STATE)
    pipeline.fit(X_train_clean, y_train)
    
    eval_result = evaluate_model(pipeline, X_test_clean, y_test, label_names=SENTIMENTS)
    results[name] = {
        'pipeline': pipeline,
        'accuracy': eval_result['accuracy'],
        'report': eval_result['report'],
        'report_dict': eval_result['report_dict'],
        'confusion_matrix': eval_result['confusion_matrix'],
        'predictions': eval_result['predictions'],
    }
    
    print(f"  Accuracy: {eval_result['accuracy']:.4f}")
    print(f"\n{eval_result['report']}")

In [ ]:
# Model comparison summary
comparison = pd.DataFrame({
    'Model': [name.upper() for name in model_names],
    'Accuracy': [results[n]['accuracy'] for n in model_names],
    'Macro F1': [results[n]['report_dict']['macro avg']['f1-score'] for n in model_names],
    'Weighted F1': [results[n]['report_dict']['weighted avg']['f1-score'] for n in model_names],
}).sort_values('Accuracy', ascending=False).reset_index(drop=True)

print("\n" + "="*60)
print("  MODEL COMPARISON SUMMARY")
print("="*60)
print(comparison.to_string(index=False))

best_model_name = comparison.iloc[0]['Model'].lower()
print(f"\n  Best model: {best_model_name.upper()} "
      f"with accuracy {comparison.iloc[0]['Accuracy']:.4f}")

# Bar chart comparison
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(comparison))
width = 0.25
ax.bar(x - width, comparison['Accuracy'], width, label='Accuracy', color='#4C72B0')
ax.bar(x, comparison['Macro F1'], width, label='Macro F1', color='#55A868')
ax.bar(x + width, comparison['Weighted F1'], width, label='Weighted F1', color='#C44E52')
ax.set_xticks(x)
ax.set_xticklabels(comparison['Model'])
ax.set_ylim(0.5, 1.02)
ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison — Sentiment Classification',
             fontweight='bold', fontsize=14)
ax.legend()
for i in range(len(comparison)):
    ax.text(i - width, comparison.iloc[i]['Accuracy'] + 0.005,
            f"{comparison.iloc[i]['Accuracy']:.3f}", ha='center', fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'results', 'model_comparison.png'),
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Select the best model for deeper analysis
best = results[best_model_name]
best_pipeline = best['pipeline']
y_pred = best['predictions']

print(f"Using best model: {best_model_name.upper()}")
print(f"Accuracy: {best['accuracy']:.4f}")

In [ ]:
# Top features (most discriminative words/bigrams) per sentiment
print("\nTop 15 Features per Sentiment (by model coefficient weight):")
print("=" * 60)

top_feats = get_top_features_per_class(best_pipeline, SENTIMENTS, top_n=15)

for sent, features in top_feats.items():
    print(f"\n  {sent.upper()}:")
    for word, weight in features:
        print(f"    {word:<25s} {weight:>8.4f}")

print("\n" + "─"*60)
print("Reasoning: The top features reveal HOW the model reasons about sentiment:")
print("  - Negative: words like 'kill', 'attack', 'crash', 'death' carry strong negative signal")
print("  - Positive: words like 'win', 'boost', 'new', 'celebrate' indicate positive events")
print("  - Neutral: factual/procedural words — the model learns by absence of valence")

---
## Step 7: Ambiguity Analysis & Logical Reasoning

We identify misclassified samples and reason about **why** the model (and VADER labeling) fails. This reveals genuine linguistic ambiguity in sentiment expression.

In [ ]:
# Get misclassified examples
misclassified_df = get_misclassified(
    X_test_clean, y_test, y_pred.tolist(), best_pipeline
)

# Also add the original (uncleaned) headlines for readability
y_test_arr = np.array(y_test)
y_pred_arr = np.array(y_pred)
mis_mask = y_test_arr != y_pred_arr
mis_originals = [X_test[i] for i in np.where(mis_mask)[0]]
if len(misclassified_df) > 0:
    misclassified_df['original_headline'] = mis_originals

print(f"Total test headlines: {len(X_test_clean):,}")
print(f"Misclassified:       {len(misclassified_df):,}")
print(f"Error rate:          {len(misclassified_df)/len(X_test_clean)*100:.2f}%")

if len(misclassified_df) > 0:
    print(f"\nMisclassification breakdown (true → predicted):")
    confusion_pairs = misclassified_df.groupby(
        ['true_label', 'predicted_label']).size().sort_values(ascending=False)
    print(confusion_pairs.to_string())

In [ ]:
# Display misclassified examples with reasoning
if len(misclassified_df) > 0:
    print("DETAILED MISCLASSIFIED EXAMPLES — AMBIGUITY ANALYSIS")
    print("=" * 80)
    n_show = min(20, len(misclassified_df))
    sample_mis = misclassified_df.sample(n=n_show, random_state=RANDOM_STATE)
    for idx, (_, row) in enumerate(sample_mis.iterrows(), 1):
        print(f"\n--- Example {idx} ---")
        print(f"Original headline: {row['original_headline']}")
        print(f"True label:        {row['true_label']}")
        print(f"Predicted label:   {row['predicted_label']}")
        if 'confidence' in row:
            print(f"Pred confidence:   {row['confidence']:.4f}")
            print(f"True label prob:   {row['true_label_prob']:.4f}")
        print()
else:
    print("No misclassified examples — perfect classification!")

In [ ]:
# Save misclassified examples to CSV
if len(misclassified_df) > 0:
    output_path = os.path.join(PROJECT_ROOT, 'results', 'misclassified_examples.csv')
    misclassified_df.to_csv(output_path, index=False)
    print(f"Misclassified examples saved to: {output_path}")
    print(f"Total: {len(misclassified_df):,} misclassified headlines")

### Ambiguity Discussion & Logical Reasoning

The misclassified examples reveal several important patterns of **sentiment ambiguity** in news headlines:

#### Common Ambiguous Patterns

1. **Negation-based ambiguity**: Headlines like *"Government denies failure"* — "denies" is negative, "failure" is negative, but the overall meaning may be neutral or defensive. The bag-of-words model cannot capture negation scope.

2. **Mixed-sentiment events**: *"Thousands flee as bushfire destroys homes"* — "destroys" is clearly negative, but the flight to safety carries a neutral/survival angle. The model sees conflicting signals.

3. **Irony and understatement**: *"Not the best day for investors"* — the literal words are mildly negative, but the meaning is strongly negative. VADER and TF-IDF both struggle with indirect expression.

4. **Neutral ↔ Negative borderline**: Factual crime/accident reporting ("Man charged with fraud") — factual tone but negative topic. The classification depends on whether sentiment is about the *topic* or the *tone*.

5. **Neutral ↔ Positive borderline**: *"Company announces new product"* — factual, but "new" carries slight positive valence. The VADER threshold (±0.05) creates boundary cases.

6. **Domain-specific language**: Sports headlines often use aggressive verbs ("crush", "dominate", "destroy") in positive contexts — e.g., *"Australia crushes England in Ashes"* is positive for Australian readers but uses violent vocabulary.

#### Logical Reasoning About Failures

| Failure Mode | Reasoning | Example |
|---|---|---|
| **Negation blindness** | BoW/TF-IDF models ignore word order; "not good" = "good" + "not" | *"no improvement in economy"* |
| **Sarcasm/irony** | Surface-level features miss pragmatic meaning | *"another brilliant plan from government"* |
| **Valence conflict** | Headline mixes positive and negative words | *"hero firefighter dies saving family"* |
| **Cultural context** | Sports/politics terms have domain-specific valence | *"opposition destroyed in debate"* |
| **VADER threshold edge** | Scores near ±0.05 are inherently unreliable | Any headline scoring ~0.04 or ~-0.04 |

#### Suggestions for Improvement

1. **Contextual embeddings**: Use BERT/RoBERTa to capture word order and negation.
2. **Better labeling**: Use human-annotated sentiment labels instead of VADER auto-labels.
3. **Multi-label sentiment**: Allow "mixed" as a fourth category for genuinely ambiguous headlines.
4. **Domain adaptation**: Train separate models for sports, politics, business subdomains.
5. **Confidence filtering**: Flag headlines with VADER scores near the thresholds for manual review.

---
## Step 8: Confusion Matrix Visualization

In [ ]:
# Confusion matrix for the best model
cm = best['confusion_matrix']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=SENTIMENTS,
            yticklabels=SENTIMENTS, ax=axes[0], linewidths=0.5,
            cbar_kws={'label': 'Count'})
axes[0].set_title(f'Confusion Matrix — {best_model_name.upper()} (Counts)',
                  fontweight='bold', fontsize=14)
axes[0].set_xlabel('Predicted Sentiment')
axes[0].set_ylabel('True Sentiment')

# Normalized (percentages)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Greens', xticklabels=SENTIMENTS,
            yticklabels=SENTIMENTS, ax=axes[1], linewidths=0.5,
            cbar_kws={'label': 'Proportion'})
axes[1].set_title(f'Confusion Matrix — {best_model_name.upper()} (Normalized)',
                  fontweight='bold', fontsize=14)
axes[1].set_xlabel('Predicted Sentiment')
axes[1].set_ylabel('True Sentiment')

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'results', 'confusion_matrix.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print("Confusion matrix saved to results/confusion_matrix.png")

In [ ]:
# Confusion matrices for ALL models side by side
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.ravel()

for i, name in enumerate(model_names):
    cm_i = results[name]['confusion_matrix']
    cm_i_norm = cm_i.astype('float') / cm_i.sum(axis=1)[:, np.newaxis]
    sns.heatmap(cm_i_norm, annot=True, fmt='.2%', cmap='Blues',
                xticklabels=SENTIMENTS, yticklabels=SENTIMENTS,
                ax=axes[i], linewidths=0.5)
    acc = results[name]['accuracy']
    axes[i].set_title(f'{name.upper()} (Acc: {acc:.3f})', fontweight='bold')
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('True')

plt.suptitle('Confusion Matrices — All Models (Normalized)',
             fontweight='bold', fontsize=16, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'results', 'all_confusion_matrices.png'),
            dpi=150, bbox_inches='tight')
plt.show()

---
## Final Report & Summary

### Model Performance

All classifiers were trained to learn the mapping from TF-IDF headline features to VADER-derived sentiment labels. Sentiment classification of short news headlines is inherently harder than topic classification because:
- Headlines are short (5–10 words), limiting contextual signal.
- Many headlines are deliberately neutral in tone (journalistic convention).
- Sentiment is often implicit rather than explicitly stated.

### Insights from Linguistic Analysis

**POS Tag Patterns:**
- Negative headlines use more strong adjectives and action verbs.
- Positive headlines feature achievement-oriented vocabulary.
- Neutral headlines are noun-heavy and entity-dense.

**Named Entity Patterns:**
- Negative headlines reference more locations (GPE) — crises are place-anchored.
- Positive headlines reference more people (PERSON) — achievements are person-anchored.
- Entity type distribution provides a secondary signal for sentiment reasoning.

### Sentiment Ambiguity Findings

The core challenge is that news headlines employ several linguistic strategies that confound simple sentiment classifiers:
1. **Negation scope** — bag-of-words models cannot parse "not guilty" vs. "guilty".
2. **Mixed valence** — tragic events with positive outcomes (rescue after disaster).
3. **Domain-specific connotation** — sports "destruction" is positive; war "destruction" is negative.
4. **Journalistic objectivity** — trained avoidance of sentiment creates many near-neutral headlines.

### Recommendations

1. **For better accuracy**: Fine-tune a transformer model (BERT) with human-annotated sentiment labels.
2. **For production**: Use confidence thresholds to flag ambiguous headlines for human review.
3. **For deeper analysis**: Separate headlines by topic domain before sentiment classification.
4. **For research**: The neutral class is the hardest — consider a 2-class (positive/negative) variant.

In [ ]:
# Final summary
print("\n" + "=" * 70)
print("  NEWS HEADLINE SENTIMENT CATEGORIZER — FINAL SUMMARY")
print("=" * 70)
print(f"\n  Dataset:     {len(df):,} sampled headlines (from ~1.2M total)")
print(f"  Date range:  {df['publish_date'].min().date()} to {df['publish_date'].max().date()}")
print(f"  Sentiments:  {SENTIMENTS}")
print(f"  Train/Test:  {len(X_train):,} / {len(X_test):,} headlines")
print(f"  Features:    TF-IDF (1,2)-grams, {X_train_tfidf.shape[1]:,} features")
print(f"\n  Best Model:  {best_model_name.upper()}")
print(f"  Accuracy:    {best['accuracy']:.4f} ({best['accuracy']*100:.2f}%)")
print(f"  Macro F1:    {best['report_dict']['macro avg']['f1-score']:.4f}")
print(f"  Weighted F1: {best['report_dict']['weighted avg']['f1-score']:.4f}")
print(f"\n  Misclassified: {len(misclassified_df):,} / {len(X_test):,} headlines")
print(f"\n  Output files:")
print(f"    - results/confusion_matrix.png")
print(f"    - results/all_confusion_matrices.png")
print(f"    - results/model_comparison.png")
print(f"    - results/sentiment_distribution.png")
print(f"    - results/sentiment_over_time.png")
print(f"    - results/pos_distribution.png")
print(f"    - results/top20_common_words.png")
print(f"    - results/headline_stats.png")
print(f"    - results/misclassified_examples.csv")
print(f"    - results/report.html  (generated in next step)")
print("\n" + "=" * 70)
print("  Project complete!")
print("=" * 70)

---
## Step 9: Generate HTML Report

Creates a self-contained HTML page showing all result visualizations in a responsive grid with explanatory captions, then opens it in the default browser.

In [ ]:
import base64
import webbrowser
from pathlib import Path

results_dir = Path(PROJECT_ROOT) / 'results'
report_path = results_dir / 'report.html'

# Define images with captions
report_images = [
    ('sentiment_distribution.png',
     'Sentiment Distribution',
     'Distribution of VADER-labeled sentiment classes (Positive / Neutral / Negative) across the sampled headlines, including a score density plot and proportion pie chart.'),
    ('top20_common_words.png',
     'Top 20 Most Common Words',
     'The 20 most frequently occurring meaningful words across all headlines after stopword removal. Reveals the dominant topics in the dataset.'),
    ('headline_stats.png',
     'Headline Length Statistics',
     'Character count distribution and the relationship between word count and character count across headlines. Most headlines are short and concise.'),
    ('sentiment_over_time.png',
     'Sentiment Trends Over Time',
     'How the proportion of positive, neutral, and negative headlines changes year by year from 2003 to 2021.'),
    ('pos_distribution.png',
     'POS Tag Proportions by Sentiment',
     'Part-of-speech tag distribution across sentiment classes. Negative headlines use more adjectives and action verbs; neutral headlines are noun-heavy.'),
    ('model_comparison.png',
     'Model Performance Comparison',
     'Accuracy, Macro F1, and Weighted F1 scores for all four classifiers (LinearSVC, Logistic Regression, Naive Bayes, Random Forest).'),
    ('confusion_matrix.png',
     'Best Model Confusion Matrix',
     f'Confusion matrix (counts and normalized) for the best-performing model ({best_model_name.upper()}) showing per-class classification accuracy.'),
    ('all_confusion_matrices.png',
     'All Models — Confusion Matrices',
     'Side-by-side normalized confusion matrices for every trained model, enabling direct comparison of per-class performance.'),
]

def img_to_base64(img_path):
    with open(img_path, 'rb') as f:
        return base64.b64encode(f.read()).decode('utf-8')

# Build image cards HTML with lightbox data attributes
image_cards = []
for i, (filename, title, caption) in enumerate(report_images, 1):
    img_path = results_dir / filename
    if img_path.exists():
        b64 = img_to_base64(img_path)
        safe_title   = title.replace('"', '&quot;')
        safe_caption = caption.replace('"', '&quot;')
        image_cards.append(f'''      <div class="card">
        <div class="card-header">
          <span class="card-num">{i}</span>
          <span class="card-title">{title}</span>
        </div>
        <div class="card-img-wrap" onclick="openLightbox(this)" title="Click to enlarge">
          <img src="data:image/png;base64,{b64}" alt="{safe_title}"
               data-title="{safe_title}" data-caption="{safe_caption}" loading="lazy" />
        </div>
        <p class="card-caption">{caption}</p>
      </div>''')
    else:
        image_cards.append(f'''      <div class="card missing">
        <div class="card-header">
          <span class="card-num">{i}</span>
          <span class="card-title">{title}</span>
        </div>
        <p class="card-caption" style="padding:1.2rem 1.35rem;">&#9888; Image not found: {filename}</p>
      </div>''')

html_content = f'''<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Sentiment Categorizer — Results Report</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link href="https://fonts.googleapis.com/css2?family=Playfair+Display:wght@400;600&family=Inter:wght@300;400;500;600&display=swap" rel="stylesheet">
<style>
  :root {{
    --bg:           #f0ede8;
    --bg-card:      #ffffff;
    --text:         #1c1917;
    --text-muted:   #78716c;
    --accent:       #1d4ed8;
    --accent-subtle:#eff6ff;
    --accent-ring:  #bfdbfe;
    --border:       #e7e5e0;
    --border-inner: #f0ede8;
    --radius:       14px;
    --radius-sm:    8px;
  }}
  *, *::before, *::after {{ margin:0; padding:0; box-sizing:border-box; }}
  body {{
    font-family: 'Inter', sans-serif;
    background: var(--bg);
    color: var(--text);
    line-height: 1.6;
    min-height: 100vh;
  }}

  /* ── Header ─────────────────────────────── */
  header {{
    background: #ffffff;
    border-bottom: 1px solid var(--border);
    padding: 3rem 2rem 2.5rem;
    text-align: center;
  }}
  .eyebrow {{
    font-size: 0.68rem;
    font-weight: 600;
    letter-spacing: 0.16em;
    text-transform: uppercase;
    color: var(--accent);
    margin-bottom: 0.8rem;
  }}
  header h1 {{
    font-family: 'Playfair Display', Georgia, serif;
    font-size: clamp(1.9rem, 4.5vw, 3.2rem);
    font-weight: 600;
    letter-spacing: -0.02em;
    color: var(--text);
    line-height: 1.15;
    margin-bottom: 0.55rem;
  }}
  .subtitle {{
    color: var(--text-muted);
    font-size: 0.92rem;
    font-weight: 300;
    margin-bottom: 2.25rem;
  }}
  .stats {{
    display: inline-flex;
    gap: 0.65rem;
    flex-wrap: wrap;
    justify-content: center;
  }}
  .stat {{
    background: var(--bg);
    border: 1px solid var(--border);
    border-radius: 999px;
    padding: 0.5rem 1.4rem;
    text-align: center;
    min-width: 108px;
  }}
  .stat .value {{
    font-size: 1.25rem;
    font-weight: 600;
    color: var(--accent);
    line-height: 1.2;
    display: block;
  }}
  .stat .label {{
    font-size: 0.62rem;
    font-weight: 600;
    letter-spacing: 0.1em;
    text-transform: uppercase;
    color: var(--text-muted);
    display: block;
    margin-top: 0.05rem;
  }}

  /* ── Main ────────────────────────────────── */
  main {{
    max-width: 1460px;
    margin: 0 auto;
    padding: 2.5rem 2rem 3.5rem;
  }}
  .section-hint {{
    font-size: 0.72rem;
    font-weight: 500;
    letter-spacing: 0.1em;
    text-transform: uppercase;
    color: var(--text-muted);
    margin-bottom: 1.5rem;
    display: flex;
    align-items: center;
    gap: 0.75rem;
  }}
  .section-hint::after {{
    content: '';
    flex: 1;
    height: 1px;
    background: var(--border);
  }}

  /* ── Grid ────────────────────────────────── */
  .grid {{
    display: grid;
    grid-template-columns: repeat(auto-fill, minmax(580px, 1fr));
    gap: 1.5rem;
  }}

  /* ── Cards ───────────────────────────────── */
  .card {{
    background: var(--bg-card);
    border: 1px solid var(--border);
    border-radius: var(--radius);
    overflow: hidden;
    box-shadow: 0 1px 2px rgba(0,0,0,.05), 0 4px 14px rgba(0,0,0,.06);
    transition: transform 0.22s ease, box-shadow 0.22s ease;
    display: flex;
    flex-direction: column;
  }}
  .card:hover {{
    transform: translateY(-3px);
    box-shadow: 0 6px 28px rgba(0,0,0,.11), 0 1px 6px rgba(0,0,0,.05);
  }}
  .card-header {{
    padding: 1rem 1.35rem 0.75rem;
    display: flex;
    align-items: center;
    gap: 0.65rem;
    border-bottom: 1px solid var(--border-inner);
  }}
  .card-num {{
    width: 24px;
    height: 24px;
    border-radius: 50%;
    background: var(--accent-subtle);
    border: 1px solid var(--accent-ring);
    color: var(--accent);
    font-size: 0.68rem;
    font-weight: 700;
    display: flex;
    align-items: center;
    justify-content: center;
    flex-shrink: 0;
  }}
  .card-title {{
    font-size: 0.875rem;
    font-weight: 600;
    color: var(--text);
    letter-spacing: -0.01em;
  }}
  .card-img-wrap {{
    padding: 1rem;
    cursor: zoom-in;
    flex: 1;
    display: flex;
    align-items: center;
    background: #fafaf9;
    position: relative;
  }}
  .zoom-hint {{
    position: absolute;
    top: 0.6rem;
    right: 0.75rem;
    font-size: 0.7rem;
    opacity: 0;
    transition: opacity 0.2s;
    pointer-events: none;
    background: rgba(255,255,255,0.92);
    padding: 0.2rem 0.5rem;
    border-radius: 4px;
    border: 1px solid var(--border);
    color: var(--text-muted);
  }}
  .card:hover .zoom-hint {{ opacity: 1; }}
  .card-img-wrap img {{
    width: 100%;
    display: block;
    border-radius: var(--radius-sm);
    border: 1px solid var(--border);
    transition: opacity 0.18s;
  }}
  .card:hover .card-img-wrap img {{ opacity: 0.93; }}
  .card-caption {{
    padding: 0.7rem 1.35rem 1.1rem;
    font-size: 0.8rem;
    color: var(--text-muted);
    line-height: 1.6;
    background: #fafaf9;
    border-top: 1px solid var(--border-inner);
  }}
  .card.missing {{ border-color: #fca5a5; background: #fff5f5; }}

  /* ── Lightbox ────────────────────────────── */
  .lightbox {{
    display: none;
    position: fixed;
    inset: 0;
    z-index: 9999;
    background: rgba(12, 11, 10, 0.82);
    backdrop-filter: blur(16px);
    -webkit-backdrop-filter: blur(16px);
    align-items: center;
    justify-content: center;
    padding: 2rem;
    cursor: zoom-out;
  }}
  .lightbox.active {{
    display: flex;
    animation: lb-fade 0.18s ease forwards;
  }}
  @keyframes lb-fade {{
    from {{ opacity: 0; }}
    to   {{ opacity: 1; }}
  }}
  .lb-inner {{
    position: relative;
    max-width: min(1280px, 92vw);
    display: flex;
    flex-direction: column;
    align-items: center;
    cursor: default;
    animation: lb-rise 0.24s cubic-bezier(0.34, 1.36, 0.64, 1) forwards;
  }}
  @keyframes lb-rise {{
    from {{ transform: scale(0.88) translateY(16px); opacity: 0; }}
    to   {{ transform: scale(1) translateY(0);       opacity: 1; }}
  }}
  .lb-inner img {{
    max-width: 100%;
    max-height: 80vh;
    display: block;
    border-radius: 10px;
    box-shadow: 0 32px 80px rgba(0,0,0,0.65), 0 4px 20px rgba(0,0,0,0.3);
    border: 1px solid rgba(255,255,255,0.10);
  }}
  .lb-meta {{
    margin-top: 1rem;
    text-align: center;
    max-width: 680px;
    padding: 0 0.5rem;
  }}
  .lb-title {{
    font-family: 'Playfair Display', serif;
    font-size: 1.1rem;
    font-weight: 600;
    color: #f5f5f4;
    margin-bottom: 0.3rem;
  }}
  .lb-caption {{
    font-size: 0.84rem;
    color: rgba(245,245,244,0.58);
    line-height: 1.6;
  }}
  .lb-hint {{
    margin-top: 0.65rem;
    font-size: 0.7rem;
    color: rgba(245,245,244,0.3);
    letter-spacing: 0.06em;
    text-transform: uppercase;
  }}
  .lb-close {{
    position: absolute;
    top: -14px;
    right: -14px;
    width: 34px;
    height: 34px;
    border-radius: 50%;
    background: #ffffff;
    border: none;
    cursor: pointer;
    display: flex;
    align-items: center;
    justify-content: center;
    box-shadow: 0 2px 14px rgba(0,0,0,0.35), 0 0 0 1px rgba(0,0,0,0.06);
    color: #1c1917;
    font-size: 0.9rem;
    line-height: 1;
    transition: background 0.15s, transform 0.18s;
    z-index: 1;
  }}
  .lb-close:hover {{
    background: #f5f5f4;
    transform: scale(1.12) rotate(90deg);
  }}

  /* ── Footer ──────────────────────────────── */
  footer {{
    background: #ffffff;
    border-top: 1px solid var(--border);
    text-align: center;
    padding: 1.5rem 2rem;
    color: var(--text-muted);
    font-size: 0.78rem;
    letter-spacing: 0.01em;
  }}

  @media (max-width: 680px) {{
    .grid {{ grid-template-columns: 1fr; }}
    main {{ padding: 1.5rem 1rem 2.5rem; }}
    header {{ padding: 2rem 1rem 2rem; }}
    .lb-close {{ top: -10px; right: -10px; width: 30px; height: 30px; font-size: 0.8rem; }}
  }}
</style>
</head>
<body>

<header>
  <p class="eyebrow">NLP &middot; Machine Learning &middot; Sentiment Analysis</p>
  <h1>News Headline Sentiment Categorizer</h1>
  <p class="subtitle">Results Report &mdash; ABC News Headlines Dataset</p>
  <div class="stats">
    <div class="stat">
      <span class="value">{len(df):,}</span>
      <span class="label">Headlines</span>
    </div>
    <div class="stat">
      <span class="value">{best_model_name.upper()}</span>
      <span class="label">Best Model</span>
    </div>
    <div class="stat">
      <span class="value">{best["accuracy"]*100:.1f}%</span>
      <span class="label">Accuracy</span>
    </div>
    <div class="stat">
      <span class="value">{best["report_dict"]["macro avg"]["f1-score"]:.3f}</span>
      <span class="label">Macro F1</span>
    </div>
    <div class="stat">
      <span class="value">{len(misclassified_df):,}</span>
      <span class="label">Misclassified</span>
    </div>
  </div>
</header>

<!-- Lightbox overlay -->
<div class="lightbox" id="lightbox">
  <div class="lb-inner" id="lb-inner">
    <button class="lb-close" id="lb-close" title="Close (Esc)">&#x2715;</button>
    <img id="lb-img" src="" alt="" />
    <div class="lb-meta">
      <p class="lb-title" id="lb-title"></p>
      <p class="lb-caption" id="lb-caption"></p>
    </div>
    <p class="lb-hint">Press Esc or click outside to close</p>
  </div>
</div>

<main>
  <p class="section-hint">Visualizations &mdash; click any chart to enlarge</p>
  <div class="grid">
{"".join(image_cards)}
  </div>
</main>

<footer>
  Automatic News Headline Sentiment Categorizer &middot;
  ABC News Headlines (~1.2M total, 2003&ndash;2021) &middot;
  {len(df):,} headlines sampled
</footer>

<script>
  const lb      = document.getElementById('lightbox');
  const lbInner = document.getElementById('lb-inner');
  const lbImg   = document.getElementById('lb-img');
  const lbTitle = document.getElementById('lb-title');
  const lbCap   = document.getElementById('lb-caption');
  const lbClose = document.getElementById('lb-close');

  function openLightbox(wrap) {{
    const img = wrap.querySelector('img');
    lbImg.src            = img.src;
    lbImg.alt            = img.getAttribute('data-title') || '';
    lbTitle.textContent  = img.getAttribute('data-title') || '';
    lbCap.textContent    = img.getAttribute('data-caption') || '';
    lb.classList.add('active');
    document.body.style.overflow = 'hidden';
  }}

  function closeLightbox() {{
    lb.classList.remove('active');
    document.body.style.overflow = '';
  }}

  lbClose.addEventListener('click', function(e) {{
    e.stopPropagation();
    closeLightbox();
  }});

  lb.addEventListener('click', function(e) {{
    if (e.target === lb) closeLightbox();
  }});

  document.addEventListener('keydown', function(e) {{
    if (e.key === 'Escape') closeLightbox();
  }});
</script>
</body>
</html>'''

# Write HTML report
report_path.write_text(html_content, encoding='utf-8')
print(f"HTML report saved to: {report_path}")
print(f"Images embedded: {sum(1 for f,_,_ in report_images if (results_dir / f).exists())} / {len(report_images)}")

# Open in default browser
webbrowser.open(report_path.as_uri())
print("Opened report in default browser.")